# Construye tu ChatPDF normativo: un pipeline RAG de punta a punta**Data Science con Python — Universidad del Pacífico — 2026-II**Jefe de práctica: Paul Melo Ramos · Profesor: Alexander QuispeRepositorio del proyecto: `https://github.com/patrickmelorr-lang/BECA-18-RAG`---## De qué va esta prácticaEn la teoría vieron cómo funciona un LLM por dentro: tokens, distribución de probabilidad,matriz de embeddings, producto punto. Hoy usamos todo eso para construir **una sola cosa quefunciona**: un asistente que responde preguntas sobre el reglamento de Beca 18 citando lapágina, y que se niega a responder cuando la respuesta no está en el documento.El documento real:| | ||---|---|| Norma | RDE N.° 033-2026-MINEDU/VMGI-PRONABEC || Páginas | 138 || Caracteres | 372,825 || Tokens estimados | ~124,000 |## El pipeline que vamos a construir```OFFLINE (una vez)  PDF ──▶ páginas ──▶ fragmentos ──▶ vectores ──▶ ChromaDBONLINE (cada pregunta)  pregunta ──▶ vector ──▶ top-k coseno ──▶ ¿supera el umbral?                                              ├─ no ──▶ "no sé"  (sin llamar al LLM)                                              └─ sí ──▶ prompt aumentado ──▶ DeepSeek                                                            └──▶ respuesta + página + costo```## La regla de la claseCada bloque termina con **un número que ustedes midieron**. No hay ninguna afirmación en estenotebook que no se pueda comprobar corriendo una celda. Si algo no se puede medir, no va.---## Índice| Bloque | Qué se mide | La lección ||---|---|---|| 1 | tokens y costo del documento | medir antes de programar || 2 | cobertura de citas de página | metadata en metadata, texto en texto || 3 | fragmentos según el tamaño de chunk | el chunking es una decisión, no un default || 4 | efecto de los prefijos del modelo e5 | leer la documentación del modelo || 5 | distancia contra similitud | Chroma no devuelve lo que crees || 6 | caída del ranking al parafrasear | la recuperación semántica es frágil || 7 | Recall@1, @3, @5 | el set de evaluación también se audita || 8 | tokens, caché y costo real | el output y el caché mandan en la factura || 9 | el pipeline completo | el motor y sus caras || 10 | el mismo pipeline en n8n | cuándo conviene no escribir código |

---# Bloque 0 — Preparar el entornoDos caminos. Elige uno.**Camino A — Google Colab.** No instalas nada en tu máquina. Todo se borra al cerrar.**Camino B — Tu laptop.** Es el que van a usar para el proyecto final, porque Streamlit y elbot de Telegram necesitan correr como procesos tuyos.La llave de DeepSeek va en los *Secrets* de Colab (ícono de llave, a la izquierda) o en unarchivo `.env` en local. **Nunca dentro de una celda.** Una llave escrita en el código y subidaa GitHub la detectan bots en segundos.

In [ ]:
# ---------- Camino A: Colab ----------# !git clone https://github.com/patrickmelorr-lang/BECA-18-RAG.git# %cd BECA-18-RAG# !pip install -q -r requirements.txt# ---------- Camino B: local ----------# git clone https://github.com/patrickmelorr-lang/BECA-18-RAG.git# cd BECA-18-RAG# python -m venv .venv && .venv\Scripts\activate      (Windows)# pip install torch --index-url https://download.pytorch.org/whl/cpu# pip install -r requirements.txtimport sys, os, time, json, refrom pathlib import Pathsys.path.insert(0, str(Path.cwd()))import numpy as npimport pandas as pdfrom src import settingscfg = settings.cargar()print("Proyecto :", cfg["proyecto"])print("Documento:", cfg["documento"]["nombre"])print("Chunking :", cfg["chunking"]["tamano"], "caracteres, solape", cfg["chunking"]["solape"])print("Embedding:", cfg["embeddings"]["modelo"])print("Modelo   :", cfg["generacion"]["modelo"])

### Por qué toda la configuración vive en `config.yaml`Abran el archivo. Ahí están las rutas, el tamaño de chunk, el modelo, la temperatura, el umbraly la tabla de precios.**La regla que se califica en el Issue:** si un número aparece dentro de `src/`, está mal.Y fíjense en un campo que casi nadie pone:```yamlprecios:  verificado_el: "2026-09-14"  fuente: "https://api-docs.deepseek.com/quick_start/pricing"```Un precio sin fecha no es auditable. Cuando dentro de tres meses alguien les pregunte de dóndesalió el presupuesto, esa línea es la respuesta.

---# Bloque 1 — Medir antes de programar## La idea, en versión de niñoAntes de mudarte, mides el sofá y mides la puerta. Nadie carga el sofá hasta el quinto piso paradescubrir arriba que no entra.Aquí el sofá es el documento y la puerta es la ventana de contexto del modelo.## Lo que vamos a responder1. ¿Cuántos tokens tiene el reglamento?2. ¿Entra completo en el modelo?3. Si entra, ¿cuánto costaría mandarlo entero en **cada** pregunta?

In [ ]:
from src import extractpaginas = extract.extraer_paginas(cfg["documento"]["ruta"])r = extract.resumen(paginas)print(f"Páginas útiles   : {r['paginas']}")print(f"Caracteres       : {r['caracteres']:,}")print(f"Palabras         : {r['palabras']:,}")print(f"Tokens estimados : {r['tokens_estimados']:,}")print()print(f"tokens por palabra: {r['tokens_estimados']/r['palabras']:.2f}")print(f"caracteres por token: {r['caracteres']/r['tokens_estimados']:.2f}")

### Lean el resultadoAlrededor de **2.3 tokens por palabra**. En inglés la relación típica es de 1.3.Esa diferencia es el **sobrecosto del español**, y no es una curiosidad académica: es una líneade presupuesto. Procesar un millón de documentos en español cuesta entre 30% y 60% más que elmismo contenido en inglés, porque el tokenizador fue entrenado mayoritariamente con inglés yparte nuestras palabras en más pedazos.En la teoría lo vieron con `desafortunadamente` → `des` + `af` + `ortun` + `adamente`.Acá lo están viendo sobre 55,000 palabras reales de una norma peruana.

In [ ]:
from src.costs import costo_usdprecios = cfg["precios"]modelo = cfg["generacion"]["modelo"]tok_doc = r["tokens_estimados"]# Escenario 1: mandar el reglamento completo en cada preguntac_completo = costo_usd(precios, modelo, tok_doc, 300)# Escenario 2: RAG, solo los fragmentos relevantestok_rag = 900c_rag = costo_usd(precios, modelo, tok_rag, 300)print(f"Documento completo ({tok_doc:>7,} tokens in): USD {c_completo:.6f} por pregunta")print(f"RAG con k=5        ({tok_rag:>7,} tokens in): USD {c_rag:.6f} por pregunta")print(f"\nRAG es {c_completo/c_rag:.0f} veces más barato.")print(f"Con 1,000 preguntas al mes: USD {c_completo*1000:.2f} contra USD {c_rag*1000:.2f}")

### La decisión que acaban de tomarEl documento **sí entra** en una ventana de 128K tokens. Podrían mandarlo completo y saltarsetodo el resto de este notebook.La pregunta correcta no es "¿cabe?" sino **"¿conviene pagarlo?"**. Y ahora tienen el número.Esta es una de las tres decisiones técnicas que les van a pedir defender en el video delproyecto. La respuesta no es "usé RAG porque es lo moderno", es "usé RAG porque mandar eldocumento completo costaba 60 veces más por consulta".

---# Bloque 2 — El bug de la cita de páginaEste bloque es el corazón de la práctica. Vamos a encontrar un error real, medirlo y arreglarlo.## La situaciónEl asistente tiene una instrucción en su system prompt:> *"Cita siempre la página de la que sacaste el dato."*Parece razonable. Y la primera versión del proyecto lo implementó así: al extraer el PDF, seinsertaba un marcador `[PAGE 12]` **dentro del texto**, al inicio de cada página, y después separtía todo en fragmentos.**Pregunta para el aula antes de correr nada:** ¿qué pasa con un fragmento que cae en la mitadde la página 12?

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter# --- ENFOQUE VIEJO: marcador dentro del texto, luego partir todo junto ---texto_completo = "\n\n".join(f"[PAGE {p['pagina']}]\n{p['texto']}" for p in paginas)splitter = RecursiveCharacterTextSplitter(    chunk_size=400, chunk_overlap=60, separators=["\n\n", "\n", ". ", " "])chunks_viejo = splitter.split_text(texto_completo)con_marcador = sum(1 for c in chunks_viejo if "[PAGE" in c)print("ENFOQUE VIEJO (marcador como texto)")print(f"  fragmentos totales        : {len(chunks_viejo):,}")print(f"  con marcador de página    : {con_marcador:,}  ({con_marcador/len(chunks_viejo)*100:.1f}%)")print(f"  SIN forma de citar página : {len(chunks_viejo)-con_marcador:,}  "      f"({(len(chunks_viejo)-con_marcador)/len(chunks_viejo)*100:.1f}%)")

### El númeroAlrededor del **9%** de los fragmentos conserva el marcador. Ese 9% son, exactamente, losprimeros fragmentos de cada página. Todos los demás quedaron huérfanos.### Lo que pasa en producción, que es peor que no citarRecuperas 5 fragmentos. Uno solo trae marcador. El modelo tiene la orden de citar, así queobedece: atribuye **toda** la respuesta a esa página.No es una cita faltante. Es una **cita confiadamente equivocada**. Un funcionario de PRONABECabre la página 42 a buscar algo que estaba en la 87, y el sistema pierde toda su credibilidad.Conecta directamente con lo que vieron en teoría: *sin la información adecuada, el LLM produceel token más probable con total seguridad*. El modelo no mintió. Hizo lo correcto con lo que ledimos. **El error fue de arquitectura, no del LLM.**### El arregloEl número de página no es texto. Es un **dato sobre** el texto. Va en metadata.Concretamente: en vez de concatenar las 138 páginas y partir el resultado, troceamos**página por página** y le colgamos el número a cada fragmento.

In [ ]:
from src import chunkingchunks = chunking.trocear(    paginas,    cfg["chunking"]["tamano"],    cfg["chunking"]["solape"],    cfg["chunking"]["separadores"],)rc = chunking.resumen(chunks)print("ENFOQUE NUEVO (página como metadata)")print(f"  fragmentos totales     : {rc['chunks']:,}")print(f"  con número de página   : {rc['con_pagina']:,}  "      f"({rc['con_pagina']/rc['chunks']*100:.1f}%)")print(f"  páginas cubiertas      : {rc['paginas_cubiertas']}")print(f"  largo promedio         : {rc['largo_promedio']} caracteres")print()c = chunks[100]print(f"Ejemplo -> id={c['id']}  pagina={c['pagina']}")print(f"  {c['texto'][:180]}...")

### Antes y después| | Marcador en el texto | Página en metadata ||---|---:|---:|| Fragmentos que pueden citar su página | ~9% | **100%** |La regla que se llevan, y que aplica a cualquier proyecto de documentos:> **Texto en el texto. Datos sobre el texto, en metadata.**Y hay un bonus: con la página en metadata pueden **filtrar** la búsqueda. "Solo el Título III","solo las páginas 100 a 138". Con el marcador embebido eso era imposible.

---# Bloque 3 — Chunking: por qué 900 y no 400## La idea, en versión de niñoCortas un libro en tiritas para poder buscar rápido. Si cortas muy chiquito, la frase quedapartida y ninguna tirita se entiende sola. Si cortas muy grande, cada tirita trae mucha cosa queno tiene nada que ver.## El parámetro `overlap`Cada tirita repite las últimas letras de la anterior. Así, una frase cortada al final de unareaparece completa al inicio de la siguiente.```sin solape:   "...el monto máximo de desembolso"  |  "asciende a S/ 8,000..."con solape:   "...el monto máximo de desembolso"  |  "de desembolso asciende a S/ 8,000..."```

In [ ]:
print(f"{'chunk':>6} {'solape':>7} {'fragmentos':>11} {'largo prom.':>12}")print("-" * 40)for tam, ov in [(400, 60), (600, 100), (900, 150), (1200, 180)]:    ch = chunking.trocear(paginas, tam, ov, cfg["chunking"]["separadores"])    rr = chunking.resumen(ch)    print(f"{tam:>6} {ov:>7} {rr['chunks']:>11,} {rr['largo_promedio']:>12}")

### Cómo se elige, de verdadNo se elige leyendo un blog. Se elige **midiendo Recall@k** con los tres valores, que es lo quehacemos en el Bloque 7.Para este documento la hipótesis es que 400 es demasiado chico: un artículo de un reglamentoperuano suele pasar los 400 caracteres, así que ese tamaño parte artículos por la mitad.Hay una pista que lo delata. La primera versión de la interfaz tenía el control de `k` puesto en**13** aunque el valor por defecto de la función era 5. Eso cuenta una historia: con 5 noencontraba la respuesta, y se subió `k` hasta que la encontrara.> **Subir `k` para tapar una recuperación mala es compensar con fuerza bruta.** Cuesta más> tokens, mete ruido en el prompt, y esconde el problema real, que estaba en el chunking.### La alternativa que casi siempre gana en documentos legalesCortar por **artículo**, no por número de caracteres. La unidad semántica ya viene marcada en eldocumento; el splitter solo tiene que respetarla.

In [ ]:
def trocear_por_articulo(paginas):    """Chunking consciente de la estructura: corta donde el documento ya cortó."""    texto = "\n".join(p["texto"] for p in paginas)    partes = re.split(r"(?=Art[íi]culo\s+\d+)", texto)    return [p.strip() for p in partes if len(p.strip()) > 80]arts = trocear_por_articulo(paginas)print(f"Fragmentos por artículo: {len(arts)}")if arts:    largos = [len(a) for a in arts]    print(f"Largo: min {min(largos)}  mediana {int(np.median(largos))}  max {max(largos)}")    print(f"\nPrimeros tres:")    for a in arts[:3]:        print(f"  [{len(a):>5} car.] {a[:90]}...")

> **Ejercicio 1.** Este documento es una resolución directoral, no un código articulado clásico,> así que el corte por `Artículo N` puede recuperar pocos fragmentos. Adapten la expresión> regular a la estructura real del documento (`Numeral`, `Sección`, `Anexo`, `TÍTULO`) y comparen> el Recall@3 contra el chunking de 900 caracteres. **Traigan el número, no la opinión.**

---# Bloque 4 — Embeddings locales y la letra chica del modelo## La idea, en versión de niñoUn token ID es un DNI: identifica, pero no describe. El número 72299846 no te dice si la personaes alta ni qué estudia.Un **embedding** es la ficha completa: una lista de números donde cada posición captura algo delsignificado. Frases parecidas dan listas parecidas.## Por qué locales y no por APILa primera versión usaba embeddings por API. Los números:| | Embeddings por API | Embeddings locales ||---|---|---|| Fragmentos a indexar | 1,483 | 570 || Límite de peticiones | sí, con esperas de 30 s por lote | no hay || Tiempo de indexado | **más de 15 minutos solo de esperas** | decenas de segundos || Costo | consume cuota | **cero** |Y la consecuencia que de verdad importa: con 15 minutos por reindexado, nadie prueba tresestrategias de chunking. Prueban una y escriben una justificación bonita. Con 30 segundos,prueban diez y traen la tabla medida.**El stack no cambia lo que se puede hacer. Cambia lo que uno efectivamente hace.**## La letra chica: los prefijosEl modelo `multilingual-e5-small` fue entrenado con prefijos obligatorios:- `"query: "` para la pregunta- `"passage: "` para el fragmento del documentoEs el equivalente local del `task_type=RETRIEVAL_QUERY` / `RETRIEVAL_DOCUMENT` que ofrecenalgunas APIs: le dice al modelo que una pregunta y un pasaje **no cumplen la misma función**,aunque ambos sean texto.Omitirlos **no da ningún error**. Simplemente empeora la recuperación, en silencio.

In [ ]:
from src import embeddingsemb = embeddings.Embebedor(cfg["embeddings"])print(f"Modelo      : {cfg['embeddings']['modelo']}")print(f"Dimensiones : {emb.dim}")# Comprobamos que los vectores salen normalizados (largo 1).# Si lo están, el producto punto ES la similitud coseno: una multiplicación# de matrices en vez de una división por normas. Por eso buscar entre un# millón de vectores toma milisegundos.v = np.array(emb.documentos(["El becario debe mantener la condición de alumno regular."])[0])print(f"Norma del vector: {np.linalg.norm(v):.6f}  (debe ser ~1.0)")

In [ ]:
def coseno(a, b):    a, b = np.asarray(a), np.asarray(b)    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))pregunta = "¿Cuánto dinero me dan al mes?"candidatos = [    "La subvención de manutención se otorga mensualmente al becario.",    "El becario debe matricularse en el número mínimo de créditos.",    "La receta del ceviche lleva limón, cebolla y ají limo.",]# --- CON prefijos (lo correcto) ---vq = emb.consulta(pregunta)vd = emb.documentos(candidatos)# --- SIN prefijos (el error silencioso) ---crudo = emb.modelo.encode([pregunta] + candidatos, normalize_embeddings=True)print(f"{'candidato':<58} {'con prefijo':>12} {'sin prefijo':>12}")print("-" * 84)for i, c in enumerate(candidatos):    print(f"{c[:56]:<58} {coseno(vq, vd[i]):>12.4f} {coseno(crudo[0], crudo[i+1]):>12.4f}")

### Cómo se lee esta tablaMiren el **orden**, no los valores absolutos. Esto es importantísimo y conecta con una lecciónde la clase de teoría:> Todas las similitudes de `e5-small` viven en una franja estrecha, típicamente entre 0.75 y> 0.90. **Un umbral como "mayor a 0.8 es relevante" no se traslada entre modelos.** Cada modelo> tiene su propia escala, y hay que calibrarla con datos propios.Por eso el `umbral_similitud` está en `config.yaml` y no dentro del código: es un parámetro quese calibra, no una constante universal.

---# Bloque 5 — El índice: distancia no es similitud## La idea, en versión de niñoEs el índice del libro. Guardas cada tirita junto a su lista de números. Cuando preguntas algo,conviertes la pregunta en números y buscas las tiritas cuyos números apuntan en la mismadirección.## La trampa que atrapa a todo el mundo**ChromaDB devuelve distancia, no similitud.** Con métrica coseno:```similitud = 1 - distancia```Un alumno ve "0.41" en pantalla y lo lee como "41% de parecido". En realidad significa**59% de parecido**. Y si el sistema ordena mal por confundirlas, el bug es silencioso.

In [ ]:
from src import index as vindexcol = vindex.abrir_coleccion(cfg["indice"])if col.count() == 0:    print("Índice vacío. Construyéndolo (toma unos segundos)...")    t0 = time.time()    vindex.indexar(col, chunks, emb, lote=cfg["embeddings"]["lote"])    print(f"Listo en {time.time()-t0:.1f} s")print(f"Fragmentos en el índice: {col.count()}")res = vindex.buscar(col, emb, "¿Cuánto es la subvención mensual?", k=3)print(f"\n{'#':<3} {'página':>7} {'distancia':>10} {'similitud':>10}")print("-" * 34)for i, f in enumerate(res, 1):    print(f"{i:<3} {f['pagina']:>7} {f['distancia']:>10.4f} {f['similitud']:>10.4f}")print(f"\nFragmento 1:\n{res[0]['texto'][:300]}...")

---# Bloque 6 — La recuperación semántica es frágil## El momento en que se entiende para qué sirve todo estoPreguntemos algo usando **palabras que no están en el documento**.

In [ ]:
consultas = [    "¿Me devuelven la plata que gasté?",          # el doc dice "reembolso"    "¿Qué pasa si jalo muchos cursos?",           # el doc dice "desaprobar créditos"    "¿Me puedo cambiar de universidad?",          # el doc dice "traslado"]for q in consultas:    r5 = vindex.buscar(col, emb, q, k=3)    print(f"\nPregunta: {q}")    for f in r5:        print(f"   pág {f['pagina']:>3} | sim {f['similitud']:.4f} | {f['texto'][:95]}...")

### Lo que acaba de pasarLa pregunta dice **"plata"** y **"gasté"**. El documento dice **"reembolso"** y**"costo de postulación"**. Cero palabras en común, y aun así el fragmento correcto aparece.`Ctrl+F` habría devuelto nada. **Esa es la diferencia entre búsqueda por palabras y búsqueda porsignificado.**## Pero ahora la mala noticiaEn la clase de teoría vieron un experimento incómodo: cambiar una sola palabra de la preguntapuede desplomar el ranking del fragmento correcto del puesto 1 al puesto 20.Comprobémoslo acá.

In [ ]:
variantes = [    "¿Cuáles son las obligaciones del becario?",      # vocabulario del documento    "¿Qué tengo que cumplir si me gano la beca?",     # paráfrasis coloquial    "¿Qué me toca hacer como becario?",               # más coloquial todavía]objetivo = Nonefor i, q in enumerate(variantes):    r10 = vindex.buscar(col, emb, q, k=10)    paginas_top = [f["pagina"] for f in r10]    if i == 0:        objetivo = paginas_top[0]   # la página que trae la formulación literal    puesto = paginas_top.index(objetivo) + 1 if objetivo in paginas_top else "fuera del top-10"    print(f"\n{q}")    print(f"   top-10 páginas: {paginas_top}")    print(f"   la página {objetivo} quedó en el puesto: {puesto}")

### La lección de ingeniería> **Ningún buscador es perfecto en el puesto 1.** Por eso RAG pasa varios fragmentos al modelo> (`k=3` o `k=5`) en vez de uno: aumenta mucho la probabilidad de que el correcto esté entre> ellos.Pero cuidado con el otro extremo, que también vieron en teoría: **más contexto no siempreayuda**. Con `k=10`, el modelo puede distraerse con nueve fragmentos irrelevantes y responderpeor que con `k=1`.No existe un `k` correcto universal. Se calibra midiendo.

---# Bloque 7 — Evaluación: los dos modos de falla de un RAG## Por qué "se ve bien" no es una métricaUn RAG puede fallar de **dos maneras distintas**, y el arreglo es opuesto en cada caso:| | Falla de **recuperación** | Falla de **generación** ||---|---|---|| Qué pasa | el fragmento correcto no entró en el top-k | entró, pero el modelo no lo usó || Síntoma | el modelo inventa | el modelo responde otra cosa del contexto || Arreglo | mejor chunking, mejor embedding, más `k` | menos ruido, menos `k`, mejor prompt |> **Subir `k` arregla la primera y empeora la segunda.** Por eso hay que saber cuál se tiene> antes de tocar nada, y por eso se evalúa en dos etapas separadas.## Las dos métricas- **Recall@k** mide la **búsqueda**: ¿está el fragmento correcto entre los k recuperados? Solo  necesita embeddings, así que corre **gratis e infinitas veces**.- **Tasa de abstención correcta** mide el **prompt**: de las preguntas fuera del documento,  ¿en cuántas el sistema se negó a responder? Esta sí consume API.

In [ ]:
import csvwith open("eval/preguntas.csv", encoding="utf-8") as f:    casos = list(csv.DictReader(f))dominio = [c for c in casos if c["tipo"] == "dominio"]fuera = [c for c in casos if c["tipo"] == "fuera_de_dominio"]print(f"Casos: {len(dominio)} de dominio, {len(fuera)} fuera de dominio\n")for k in (1, 3, 5):    aciertos = 0    filas = []    for c in dominio:        esperadas = {int(x) for x in c["paginas_esperadas"].split("|") if x}        recuperadas = {f["pagina"] for f in vindex.buscar(col, emb, c["pregunta"], k=k)}        ok = bool(esperadas & recuperadas)        aciertos += ok        filas.append((("OK " if ok else "NO "), c["pregunta"][:46], sorted(recuperadas)))    print(f"Recall@{k} = {aciertos/len(dominio):.2f}  ({aciertos}/{len(dominio)})")    for est, q, rec in filas:        print(f"   {est} {q:<48} {rec}")    print()

### Los resultados reales de este proyecto| | Recall ||---|---:|| @1 | 0.33 || @3 | 0.67 || @5 | 0.83 |**Cómo se lee esta curva.** Sube fuerte con `k`. Eso significa que el buscador **sí encuentra lazona correcta del documento, pero no la pone primera**. Es el síntoma clásico de un corpus conmucho texto repetitivo: una resolución directoral está llena de fórmulas legales idénticas("de conformidad con", "según lo establecido en"), y esos fragmentos se parecen entre sí más delo que se parecen a una pregunta concreta.Diagnóstico: el problema está en la **recuperación**, no en el LLM. Cambiar de DeepSeek acualquier otro modelo no movería estos números ni un punto.## La parte que casi nadie enseña: auditar el propio set de evaluaciónUna pregunta falla con **todos** los valores de k: *"¿Me reembolsan gastos?"*. El set dice que larespuesta está en la página 39 y el buscador nunca la trae.Antes de culpar al buscador, revisemos la etiqueta.

In [ ]:
# Buscamos a mano dónde aparece realmente la palabra en el documentofor p in paginas:    for m in re.finditer(r"reembols\w*", p["texto"], re.I):        a, b = max(0, m.start()-150), m.end()+150        print(f"[página {p['pagina']}] ...{p['texto'][a:b]}...\n")

### Lo que revela la auditoríaEn las 138 páginas, "reembolso" aparece **una sola vez**, y se refiere a algo muy específico: ladevolución del **costo de postulación** para quienes resulten becarios en una universidadpública.O sea: la pregunta del set estaba mal planteada. *"¿Me reembolsan gastos?"* sugiere una políticageneral de reembolsos que **este documento no tiene**. No hay un fragmento correcto querecuperar, porque la pregunta no tiene respuesta en el corpus.> **La lección:** un set de evaluación mal etiquetado te hace optimizar hacia el lugar> equivocado. Habrías pasado horas ajustando el chunking para arreglar una pregunta que nunca> estuvo bien formulada.>> **El set de evaluación también se audita.** Es parte del trabajo, no un paso previo.> **Ejercicio 2.** Reemplacen esa pregunta por *"¿Me devuelven el costo de postulación?"*,> vuelvan a correr la evaluación y reporten el nuevo Recall@3. Después agreguen 20 preguntas> más, verificando **a mano** en qué página está cada respuesta. Un set de 30 preguntas bien> etiquetadas vale más que uno de 100 hechas a ojo.

---# Bloque 8 — Generación: el umbral, el prompt y la factura## El corto circuito: no preguntar cuando no hay con qué responderAntes de llamar al modelo, el sistema revisa la similitud del mejor fragmento. Si no llega alumbral, responde "no sé" **sin gastar una sola llamada**.Dos beneficios al precio de uno: ahorra dinero y elimina la principal vía de invención. Unmodelo sin contexto útil no duda: alucina con total seguridad.## El system prompt que sostiene todo```1. Responde ÚNICAMENTE con información del CONTEXTO entregado.2. Cita siempre la página, en formato (pág. N).3. Si el contexto no contiene la respuesta, responde exactamente:   "No encuentro eso en el reglamento que tengo cargado."4. Máximo 5 oraciones.```La regla 3 es la que se prueba con las preguntas fuera de dominio. Si el sistema responde"París" a *"¿cuál es la capital de Francia?"*, el prompt no se está respetando, y mañana va ainventar un monto de subvención.

In [ ]:
from src import rag_coremotor = rag_core.MotorRAG()r1 = motor.responder("¿Cuál es el monto de la subvención económica?", k=5)print("RESPUESTA:")print(r1["respuesta"])print(f"\nPáginas citadas: {sorted({f['pagina'] for f in r1['fuentes']})}")print(f"Similitud del top-1: {r1['fuentes'][0]['similitud']}")print(f"Tokens: {r1['tokens_in']} entrada (+{r1['tokens_in_cache']} de caché) / {r1['tokens_out']} salida")print(f"Costo : USD {r1['costo_usd']:.8f}")print(f"Latencia: {r1['latencia_s']} s")

In [ ]:
# La prueba de fuego: una pregunta que el modelo SÍ sabe responder,# pero que no está en el documento.r2 = motor.responder("¿Cuál es la capital de Francia?")print("RESPUESTA:", r2["respuesta"])print("¿Se abstuvo?:", r2["abstuvo"])print("Motivo     :", r2.get("motivo") or "(llamó al modelo y este se negó)")print(f"Costo      : USD {r2['costo_usd']:.8f}")

### Si esa celda responde "París", el sistema está rotoNo es un detalle menor. Significa que el modelo ignora sus instrucciones, y un sistema queignora instrucciones con una pregunta inofensiva las va a ignorar con una pregunta cara.## El caché de prompt: datos reales de este proyectoDel log de costos del repositorio, dos llamadas **idénticas** consecutivas:| | Tokens entrada | De caché | Tokens salida | Costo | Latencia ||---|---:|---:|---:|---:|---:|| 1.ª llamada | 3,918 | 0 | 500 | USD 0.00088770 | 3.02 s || 2.ª llamada | 3,918 | 3,712 (95%) | 500 | **USD 0.00034204** | 3.33 s |**61% menos de costo**, misma respuesta. El caché cobra los tokens repetidos a una fracción delprecio.La condición para que funcione: lo que se repite (system prompt, instrucciones, esquema) debe ir**al inicio** del prompt y ser **byte a byte idéntico**. Si le metes la fecha y hora al inicio,rompes el caché en cada llamada sin darte cuenta.Acumulado real de 31 llamadas exitosas del proyecto: 77,164 tokens de entrada, 11,532 de salida,**USD 0.027 en total**. Menos de tres centavos de dólar por toda la fase de pruebas.

In [ ]:
# La contabilidad de tu propia sesiónprint(motor.contador.totales())# Y el histórico completo, agregado por modelo y operaciónfrom src.costs import reportetry:    print()    print(reporte(cfg["logs"]["costos"]))except Exception as e:    print("(aún no hay historial)", e)

---# Bloque 9 — El motor y sus caras## La arquitectura en una frase```                      ┌──────────────┐  notebook      ─────▶│              │  Streamlit     ─────▶│  rag_core    │──▶ {respuesta, fuentes, tokens, costo, latencia}  bot Telegram  ─────▶│  (el motor)  │  n8n           ─────▶│              │                      └──────────────┘````rag_core.py` expone **una sola función pública**:```pythonresponder(pregunta, k=5, temperature=0.1, modelo="deepseek-flash") -> dict```Las tres interfaces son adaptadores de entre 20 y 200 líneas. Ninguna sabe qué hay adentro.## La regla, y cómo se verifica> `rag_core.py` **no importa** `streamlit` ni nada de Telegram.Se comprueba con una línea. Si devuelve algo, la arquitectura está mal:```bashgrep -nE "^\s*(import|from)\s+(streamlit|telegram)" src/rag_core.py```## Por qué importa1. **Se puede testear.** Un módulo que importa Streamlit no corre en un test automático.2. **Se puede cambiar de cara sin tocar la lógica.** Agregar Telegram costó media hora, no un día.3. **Se puede reusar.** El mismo `rag_core.py` sirve para el proyecto de ustedes cambiando el PDF.

In [ ]:
# Las tres caras, desde acá:##   streamlit run app.py     ->  interfaz web local, con controles de k y temperature#   python bot.py            ->  bot de Telegram por polling, sin URL pública#   este notebook            ->  motor.responder(...)## Los tres escriben en el MISMO logs/costos.csv. El gasto se acumula junto.for q in ["¿Qué modalidades de beca existen?",          "¿En qué casos se suspende la beca?"]:    r = motor.responder(q, k=5)    print(f"\nP: {q}")    print(f"R: {r['respuesta'][:220]}...")    print(f"   páginas {sorted({f['pagina'] for f in r['fuentes']})} | "          f"USD {r['costo_usd']:.8f} | {r['latencia_s']}s")

---# Bloque 10 — El mismo pipeline en n8n, sin escribir código## Qué es n8n, en versión de niñoEs como armar el pipeline con bloques de Lego que se conectan con cables, en vez de escribirlo.Cada bloque hace una cosa: leer un PDF, partirlo, convertirlo en vectores, guardarlo, buscar,preguntarle al modelo.## Para qué sirve saber estoPorque en una empresa peruana real, la persona que necesita este asistente muchas veces **no esprogramadora**. Y porque un pipeline en n8n se muestra en una pantalla y se entiende en treintasegundos, mientras que un repositorio hay que leerlo.También porque n8n trae gratis lo que a ustedes les costaría escribir: reintentos, programaciónpor cron, conexión con Telegram, Google Drive, correo y cientos de servicios más.## El mapeo exacto: tu código ↔ los nodos de n8n### Flujo 1 — Indexado (equivale a `build_index.py`)| Tu módulo | Nodo de n8n | Qué configuras ||---|---|---|| disparo manual | **Manual Trigger** o **Schedule Trigger** | cron si quieres reindexado automático || `extract.py` | **Extract from File** (modo PDF) | el archivo de entrada || `chunking.py` | **Default Data Loader** + **Recursive Character Text Splitter** | `chunk size` 900, `overlap` 150, y los **metadatos** (la página) || `embeddings.py` | **Embeddings Ollama** (local) o **Embeddings OpenAI** | el modelo de embeddings || `index.py` (insertar) | **Vector Store** con operación *Insert Documents* | Simple Vector Store para probar; Qdrant para algo serio |### Flujo 2 — Consulta (equivale a `rag_core.responder`)| Tu módulo | Nodo de n8n | Qué configuras ||---|---|---|| `bot.py` | **Telegram Trigger** | el token del bot || `index.buscar` | **Vector Store** con operación *Retrieve* | `Top K` = tu `k` || `rag_core` (prompt) | **Question and Answer Chain** o **AI Agent** | el system prompt con las reglas de citación || DeepSeek | **OpenAI Chat Model** con `base_url` de DeepSeek | modelo, `temperature`, `max tokens` || `bot.enviar` | **Telegram → Send Message** | el `chat_id` viene del trigger || `costs.py` | **Google Sheets → Append** o **Postgres** | una fila por llamada |### El diagrama```FLUJO 1 (una vez)  [Manual Trigger] → [Extract from File] → [Default Data Loader]                                                   ↑                              [Recursive Character Text Splitter]                                                   ↓                                         [Vector Store: Insert]                                                   ↑                                          [Embeddings model]FLUJO 2 (cada mensaje)  [Telegram Trigger] → [Vector Store: Retrieve] → [Q&A Chain] → [Telegram: Send Message]                                    ↑                  ↑                    ↓                           [Embeddings model]   [Chat Model]      [Sheets: Append log]```## Lo que n8n te da y lo que te quita| | Código Python | n8n ||---|---|---|| Tiempo hasta el primer demo | horas | **minutos** || Reintentos, cron, conectores | los escribes | **incluidos** || Se lo muestras a un gerente | cuesta | **se entiende solo** || Control fino del chunking | **total** | limitado a lo que exponga el nodo || Medir Recall@k sobre 30 preguntas | **natural** | incómodo || Versionar y revisar cambios en Git | **natural** | un JSON gigante, el diff es ilegible || Tests automáticos | **natural** | no realmente |> **La conclusión honesta:** n8n es excelente para **prototipar y para orquestar**. El código es> mejor para **medir y para optimizar**. En la práctica profesional se usan los dos: prototipas> en n8n para validar que la idea sirve, y reescribes en código la parte que hay que afinar.>> En el proyecto del curso pedimos código porque lo que se califica es precisamente la parte que> n8n no deja hacer bien: medir Recall@k, auditar el set de evaluación, y justificar el chunking> con un número.> **Ejercicio 3.** Levanten n8n (`npx n8n` o Docker), armen el Flujo 2 con el Simple Vector> Store, y comparen dos cosas contra su versión en Python: el tiempo que les tomó construirlo, y> si pueden reproducir el Recall@3 que midieron en el Bloque 7. La segunda respuesta es la> interesante.

---# Cierre — las ocho cosas que te llevas1. **Mide antes de programar.** El documento tiene ~124,000 tokens; mandarlo completo cuesta 60   veces más por pregunta que RAG. Esa es la justificación, no "porque es lo moderno".2. **Texto en el texto, datos sobre el texto en metadata.** El marcador de página embebido   dejaba al 91% de los fragmentos sin poder citar, y el modelo citaba páginas equivocadas con   total seguridad.3. **El chunking es una decisión medible.** Y subir `k` para tapar una recuperación mala es   fuerza bruta que cuesta tokens y mete ruido.4. **Lee la documentación de tu modelo de embeddings.** Omitir los prefijos `query:` y   `passage:` no da error: solo empeora los resultados, en silencio.5. **Distancia no es similitud.** Y los umbrales no se trasladan entre modelos.6. **Un RAG falla de dos maneras opuestas.** Si Recall@3 es bajo, el problema es la búsqueda, no   el LLM. Cambiar de modelo generador no arregla una búsqueda mala.7. **El set de evaluación también se audita.** Una etiqueta mal puesta te hace optimizar hacia   el lugar equivocado durante horas.8. **El motor y sus caras son cosas distintas.** `rag_core.py` no sabe que existen Streamlit,   Telegram ni n8n. Por eso las tres funcionan.---# Tu proyectoApliquen exactamente este pipeline a un documento de su dominio. Requisitos mínimos:- [ ] Diagrama del pipeline en el `README.md`, en Mermaid, con el camino offline separado del online- [ ] `config.yaml` sin un solo número dentro de `src/`- [ ] Página o sección como **metadata**, y toda respuesta con su cita- [ ] Dos estrategias de chunking comparadas **con Recall@k**, no con argumentos- [ ] `eval/preguntas.csv` con al menos 30 preguntas, incluidas 5 fuera de dominio- [ ] `logs/costos.csv` con una fila por llamada, y el costo del informe saliendo de ahí- [ ] Umbral de similitud calibrado con sus datos, no copiado de este notebook- [ ] Streamlit local funcionandoY la pregunta que les van a hacer en la sustentación:> **"¿Tu RAG falla por recuperación o por generación? Muéstrame el número que lo prueba."**